# Искусственный интеллект — занятие 3
### Метод опорных векторов (SVM)

**Логика занятия:** как провести разделяющую линию наилучшим образом → margin и опорные векторы → ядро (kernel) и нелинейное разделение → сравнение с деревом решений → переобучение и отложенные выборки → метрики качества (precision, recall, F1).

## 1. Идея метода опорных векторов

Есть два класса точек, между ними нужно провести разделяющую линию. Подходящих линий много — какую выбрать?

**Метод опорных векторов** выбирает ту, у которой полоса (**margin**) вокруг неё максимально широка и не содержит точек. Точки, ближайшие к границе, называются **опорными векторами** — именно они задают положение границы. Максимизация margin даёт лучшую обобщающую способность: граница проходит «по центру» между классами.

## 2. Датасет и визуализация границ

Сгенерируем три облака точек (`make_blobs`) и посмотрим, как разделяют данные дерево решений и SVM. Для визуализации областей используем `plot_decision_regions` из `mlxtend`.

In [ ]:
import sys
!{sys.executable} -m pip install mlxtend

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
import graphviz
import matplotlib.pyplot as plt
from mlxtend.plotting import plot_decision_regions
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [ ]:
x, y = make_blobs(n_samples=1000, centers=3, cluster_std=3, n_features=2, random_state=42)
plt.scatter(x[:, 0], x[:, 1], c=y)
plt.title('Исходные данные make_blobs')
plt.show()

xtr, xtest, ytr, ytest = train_test_split(x, y)

clf = SVC(kernel='linear')
clf.fit(xtr, ytr)
print(f"score test {clf.score(xtest, ytest):.3f}, train {clf.score(xtr, ytr):.3f}")
plot_decision_regions(x, y, clf=clf, legend=2)
plt.title('SVM с линейным ядром')
plt.show()

## 3. Ядро и нелинейное разделение

Разделяющая поверхность задаётся уравнением `⟨w, x⟩ = c`. Чтобы получать нелинейные границы, векторы `x` заменяют на нелинейную функцию `φ(x)`: `⟨w, φ(x)⟩ = c`. Эта функция называется **ядром (kernel)**. Если ядро линейно (`φ(x) = x`), граница — прямая; сложное ядро превращает её в кривую.

По умолчанию `SVC` использует **RBF-ядро** (радиальная базисная функция), поэтому границы получаются кривыми. Указав `kernel='linear'` явно, получаем прямую.

Дерево решений, в отличие от SVM, режет пространство только линиями, перпендикулярными осям координат, поэтому на волнистых данных даёт ступенчатую границу.

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(x, y)
plot_decision_regions(x, y, clf=dt, legend=2)
plt.title('Дерево решений')
plt.show()

In [ ]:
dot_data = tree.export_graphviz(dt, out_file=None)
graph = graphviz.Source(dot_data)
graph.render("outputs/mytree10")

## 4. Переобучение, недообучение и отложенные выборки

У модели есть гиперпараметр (например, `max_depth` дерева). Теория говорит:

- на **обучающей** выборке ошибка при росте сложности всегда падает до нуля;
- на **тестовой** ошибка сначала падает, доходит до минимума, а затем снова растёт.

Минимум тестовой ошибки — граница между **недообучением** и **переобучением**. Тестовую выборку нельзя использовать для подбора гиперпараметра, иначе она «протечёт» в модель. Для выбора используют:

- **валидационную** выборку: данные делят на три части — обучающую, валидационную и тестовую; по валидации выбирают гиперпараметр, по тесту считают итоговое качество один раз;
- **кросс-валидацию**.

Обучающая выборка всегда даёт качество 1 — это не повод радоваться.

In [ ]:
xtr, xtest, ytr, ytest = train_test_split(x, y)
xtr, xval, ytr, yval = train_test_split(xtr, ytr)

g = []
for md in range(1, 50):
    clf = DecisionTreeClassifier(max_depth=md)
    clf.fit(xtr, ytr)
    g.append([md, 1 - clf.score(xval, yval)])
g = np.array(g)
opt = np.argmin(g[:, 1])

plt.plot(g[:, 0], g[:, 1])
plt.xlabel('max_depth')
plt.ylabel('ошибка на валидации')
plt.title('Подбор глубины дерева')
plt.show()

clf = DecisionTreeClassifier(max_depth=int(g[opt, 0]))
clf.fit(xtr, ytr)
print('оптимальная глубина:', int(g[opt, 0]),
      '| accuracy на тесте:', round(clf.score(xtest, ytest), 4))

## 5. Метрики качества

**Accuracy** (доля правильных ответов) — плохая метрика при дисбалансе классов. Если 95% объектов — кошки, модель «всё — кошка» даёт accuracy 0.95, но бесполезна.

Другие метрики:

- **Precision** — доля верных среди предсказанных положительных;
- **Recall** — доля найденных среди всех реально положительных;
- **F1** — гармоническое среднее precision и recall: `F1 = 2·P·R / (P + R)`;
- **Macro average F1** — F1, усреднённый по классам независимо от их размера. Это главная величина в `classification_report`: она устойчива к дисбалансу.

**Support** — число объектов каждого класса. По нему определяют, сбалансирован ли датасет: если отношение максимального класса к минимальному больше примерно 3:1, датасет несбалансированный и использовать `accuracy` нельзя.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

yp = clf.predict(xtest)
print(classification_report(ytest, yp))

In [ ]:
print('Confusion matrix:')
print(confusion_matrix(ytest, yp))

---

**Итог занятия 3.** Разобрали метод опорных векторов: максимизацию margin, ядро и нелинейные границы, сравнили с деревом решений. Разобрали переобучение, недообучение и роль обучающей, валидационной и тестовой выборок. Ввели метрики качества: precision, recall, F1 и macro average F1, а также понятие дисбаланса классов.